# Colab Setup

Mount Drive, clone or enter the repository, install dependencies, and restore the HAM10000 Drive bundle.

Run this notebook first in Colab. It restores the uploaded `ham10000_colab_bundle.tar` into the repo root so the later thin runner notebooks can use the same local paths as the scripts.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/KasimDeliaci/dl-midterm-project.git'
REPO_ROOT = Path('/content/dl-assignment')
DRIVE_BUNDLE = Path('/content/drive/MyDrive/ham10000_colab_bundle.tar')

if not (REPO_ROOT / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
print('Repo root:', Path.cwd())
print('Drive bundle exists:', DRIVE_BUNDLE.exists(), DRIVE_BUNDLE)

In [ ]:
!python -m pip install -q uv
!uv sync

In [ ]:
required_paths = [
    REPO_ROOT / 'data/raw/ham10000',
    REPO_ROOT / 'data/metadata/HAM10000_metadata.csv',
    REPO_ROOT / 'data/splits/train.csv',
    REPO_ROOT / 'data/splits/val.csv',
    REPO_ROOT / 'data/splits/test.csv',
]

if not all(path.exists() for path in required_paths):
    if not DRIVE_BUNDLE.exists():
        raise FileNotFoundError(f'Missing Drive bundle: {DRIVE_BUNDLE}')
    subprocess.run(['tar', '--warning=no-unknown-keyword', '-xf', str(DRIVE_BUNDLE), '-C', str(REPO_ROOT)], check=True)

missing = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
if missing:
    raise RuntimeError(f'Missing required dataset paths after extraction: {missing}')

print('HAM10000 bundle is ready in the repo root.')
for path in required_paths:
    print(path.relative_to(REPO_ROOT))

In [ ]:
from pathlib import Path

!nvidia-smi || true

root = Path('/content/dl-assignment')
for split in ['train', 'val', 'test']:
    split_path = root / 'data/splits' / f'{split}.csv'
    with split_path.open() as handle:
        print(split, sum(1 for _ in handle) - 1)